# Preprocessing

**Tujuan**: Membersihkan dan menormalisasi teks ulasan e-commerce sebelum modelling.

**Pipeline:**
1. `normalize_repetitive_chars` — normalisasi karakter berulang
2. `normalize_slang` — normalisasi slang (kamus alay)
3. `emoji_to_text` — konversi emoji ke teks
4. `tokenize` — tokenisasi NLTK `word_tokenize`
5. `remove_stopwords` — hapus stopwords
6. `pos_tag` — POS tagging rule-based
7. `stem` — stemming Sastrawi
8. `handle_negation` — penanganan negasi

**Output:**
- `data/intermediate/preprocessed_reviews.csv` — kolom: `Customer Review`, `text_processed`, `Sentiment`, `Emotion`
- `data/intermediate/slang_dict.joblib` — kamus slang untuk reuse di modelling

In [1]:
import pandas as pd
import re
import time
import warnings
import os
from pathlib import Path
warnings.filterwarnings('ignore')

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

import emoji
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
import joblib

RANDOM_STATE = 42

### Load Data

In [2]:
df_path = Path("../data/raw/ecommerce_reviews.csv")

if not os.path.exists(df_path):
    df = pd.read_csv("https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/59cda965-7204-45c5-b4ca-29dc27392cf7")
    df_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(df_path, index=False)
else:
    df = pd.read_csv(df_path)

df["text"] = df["Customer Review"].copy()
df["text-original"] = df["Customer Review"].copy()

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nSentiment distribution:\n{df['Sentiment'].value_counts()}")
print(f"\nEmotion distribution:\n{df['Emotion'].value_counts()}")

Total rows: 5400
Columns: ['Category', 'Product Name', 'Location', 'Price', 'Overall Rating', 'Number Sold', 'Total Review', 'Customer Rating', 'Customer Review', 'Sentiment', 'Emotion', 'text', 'text-original']

Sentiment distribution:
Sentiment
Negative    2821
Positive    2579
Name: count, dtype: int64

Emotion distribution:
Emotion
Happy      1770
Sadness    1202
Fear        920
Love        809
Anger       699
Name: count, dtype: int64


> Dataset 5.400 ulasan produk ini menunjukkan dominasi sentimen negatif (52,2%) yang persis terdiri dari emosi Sedih, Takut, dan Marah, sementara sentimen positif tercermin dari emosi Bahagia dan Cinta, dengan emosi Bahagia menjadi yang paling sering muncul secara individual meskipun sentimen negatif lebih unggul secara keseluruhan.

### Load Kamus Alay

In [3]:
kamus_path = Path("../data/util/colloquial-indonesian-lexicon.csv")

if not os.path.exists(kamus_path):
    kamus_df = pd.read_csv("https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv")
    kamus_path.parent.mkdir(parents=True, exist_ok=True)
    kamus_df.to_csv(kamus_path, index=False)
else:
    kamus_df = pd.read_csv(kamus_path)

kamus_valid = kamus_df[kamus_df["In-dictionary"] == 1]
print(f"Total entri: {len(kamus_df)}")
print(f"Entri valid (In-dictionary=1): {len(kamus_valid)}")

slang_dict = {}
for _, row in kamus_valid.iterrows():
    slang = str(row["slang"]).strip().lower()
    formal = str(row["formal"]).strip().lower()
    if slang in slang_dict:
        if len(formal) < len(slang_dict[slang]):
            slang_dict[slang] = formal
    else:
        slang_dict[slang] = formal

print(f"Unique slang entries in dictionary: {len(slang_dict)}")

Total entri: 15006
Entri valid (In-dictionary=1): 13722
Unique slang entries in dictionary: 3451


> Dari 15.006 entri kamus alay, hanya 3.451 istilah unik yang lolos validasi setelah menyaring 13.722 entri dengan status "In-dictionary=1", dan pemetaan dilakukan dengan memprioritaskan bentuk formal yang lebih pendek untuk setiap kata slang yang duplikat.

### Preprocessing Functions

In [4]:
def normalize_repetitive_chars(text: str) -> str:
    text = re.sub(r"(a)\1{2,}", "a", text)
    text = re.sub(r"(i)\1{2,}", "i", text)
    text = re.sub(r"(u)\1{2,}", "u", text)
    text = re.sub(r"(e)\1{2,}", "e", text)
    text = re.sub(r"(o)\1{2,}", "o", text)
    text = re.sub(r"([^aiueo])\1{2,}", r"\1", text)
    return text

def normalize_slang(text: str, slang_dict: dict) -> str:
    words = text.split()
    normalized = []
    for w in words:
        w_lower = w.lower()
        if w_lower in slang_dict:
            normalized.append(slang_dict[w_lower])
        elif w_lower.rstrip(".,!?;:") in slang_dict:
            punct = w[len(w_lower.rstrip(".,!?;:")):]
            normalized.append(slang_dict[w_lower.rstrip(".,!?;:")] + punct)
        else:
            normalized.append(w)
    return " ".join(normalized)

def tokenize(text: str) -> list:
    return word_tokenize(text)

def remove_stopwords(tokens: list) -> list:
    stop_words = set(stopwords.words('indonesian'))
    return [t for t in tokens if t.lower() not in stop_words]

def pos_tag(tokens: list) -> list:
    konjungsi = {'dan', 'atau', 'tetapi', 'namun', 'sedangkan', 'serta',
                 'karena', 'sehingga', 'maka', 'lalu', 'kemudian',
                 'setelah', 'sebelum', 'ketika', 'sementara', 'walaupun',
                 'meskipun', 'jika', 'kalau', 'apabila', 'bahwa'}
    preposisi = {'di', 'ke', 'dari', 'pada', 'dengan', 'untuk', 'bagi',
                 'oleh', 'tentang', 'seperti', 'sebagai', 'tanpa', 'dalam',
                 'antara', 'menurut', 'sampai', 'hingga'}
    hasil = []
    for token in tokens:
        t = token.lower()
        if re.match(r'^[.,!?;:()\[\]{}\"\'\-]$', token):
            hasil.append((token, 'PUNCT'))
        elif re.match(r'^[0-9,.\-]+$', token):
            hasil.append((token, 'NUM'))
        elif t in konjungsi:
            hasil.append((token, 'CONJ'))
        elif t in preposisi:
            hasil.append((token, 'ADP'))
        elif re.match(r'^(me|men|meng|meny|mem|di|ber|bel|ter|per)', t):
            hasil.append((token, 'VERB'))
        elif re.match(r'^(pe|pen|pem|peng|ke)', t) or t.endswith('an') or t.endswith('kan'):
            hasil.append((token, 'NOUN'))
        elif t.endswith('i'):
            hasil.append((token, 'VERB'))
        else:
            hasil.append((token, 'NOUN'))
    return hasil

def stem(tokens: list, stemmer) -> list:
    return [stemmer.stem(t) for t in tokens]

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def emoji_to_text(text: str) -> str:
    return emoji.demojize(text, language='id')

def handle_negation(tokens: list, neg_words: set = None) -> list:
    if neg_words is None:
        neg_words = {
            "tidak", "bukan", "belum", "tak", "ngga",
            "gak", "ga", "tdk", "enggak", "nggak",
            "kagak", "ndak", "ngg"
        }
    result = []
    negate = False
    for w in tokens:
        w_clean = w.lower().strip(".,!?")
        if w_clean in neg_words:
            result.append(w)
            negate = True
        elif negate:
            result.append("NEG_" + w)
            negate = False
        else:
            result.append(w)
    return result

def full_pipeline(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return text
    t = text
    t = normalize_repetitive_chars(t)
    t = normalize_slang(t, slang_dict)
    t = emoji_to_text(t)
    tokens = tokenize(t)
    tokens = remove_stopwords(tokens)
    pos_tags = pos_tag(tokens)
    tokens = stem(tokens, stemmer)
    tokens = handle_negation(tokens)
    return " ".join(tokens)

> Fungsi:
1. `normalize_repetitive_chars` menangani pengulangan huruf vokal dan konsonan berlebih
2. `normalize_slang` memetakan kata tidak baku ke bentuk formal dari kamus yang telah divalidasi
3. `tokenize` memecah teks menjadi token
4. `remove_stopwords` membuang kata hubung dan kata umum bahasa Indonesia
5. `pos_tag` memberikan label part-of-speech sederhana berbasis aturan untuk konjungsi, preposisi, verba, nomina, dan tanda baca
6. `stem` mereduksi kata menjadi bentuk dasar menggunakan pustaka Sastrawi
7. `emoji_to_text` mengonversi emoji menjadi representasi teks
8. `handle_negation` menambahkan prefiks "NEG_" pada kata setelah kata negasi untuk mempertahankan makna negatif
`full_pipeline` menggabungkan seluruh langkah tersebut menjadi satu alur pemrosesan teks yang siap digunakan untuk analisis sentimen dan emosi.

### Apply Preprocessing Pipeline

In [5]:
print("Applying preprocessing pipeline to all text...")
tic = time.time()
df['text_processed'] = df['Customer Review'].fillna('').apply(full_pipeline)
toc = time.time()
print(f"Preprocessing complete: {len(df)} texts processed in {toc - tic:.2f}s")

Applying preprocessing pipeline to all text...
Preprocessing complete: 5400 texts processed in 776.76s


> Proses pra-pemrosesan teks berhasil menyelesaikan 5.400 data dalam waktu 12,9 menit, dengan rata-rata kecepatan sekitar 0,14 detik per teks.

### Save Output

In [6]:
intermediate_dir = Path("../data/intermediate")
intermediate_dir.mkdir(parents=True, exist_ok=True)

out = pd.DataFrame({
    'Customer Review': df['Customer Review'],
    'text_processed': df['text_processed'],
    'Sentiment': df['Sentiment'],
    'Emotion': df['Emotion']
})
out.to_csv(intermediate_dir / "preprocessed_reviews.csv", index=False)
print(f"Saved: {intermediate_dir / 'preprocessed_reviews.csv'}")

joblib.dump(slang_dict, intermediate_dir / "slang_dict.joblib")
print(f"Saved: {intermediate_dir / 'slang_dict.joblib'}")

Saved: ..\data\intermediate\preprocessed_reviews.csv
Saved: ..\data\intermediate\slang_dict.joblib
